In [1]:
from glob import glob 

import numpy as np
import pandas as pd
from sklearn import preprocessing

from model import generics
from input import input
import config

In [2]:
experiment_id = 'hourwindspeed'
fold, _ = generics.format_names(experiment_id, '', '')

In [3]:
df_execs = pd.DataFrame()
for pth in glob(fold+'*'):
    model_name = pth.split('_')[-1].split('.pk')[0]
    serie_name = pth.split('/')[-1].split('_')[0]
    df_execs = pd.concat(
        [
            pd.DataFrame({'model_name': [model_name], 'serie_name': [serie_name], 'pth': [pth]}),
            df_execs
        ]
    )

In [5]:
def open_execs(exec_pkl, model_name, serie_name, shape_ts):
    df_model = pd.DataFrame()
    for n, p in enumerate(exec_pkl):
        p['experiment'].model = None
        train_predict = list(p['experiment'].metrics_results['train_predict'])
        val_predict = list(p['experiment'].metrics_results['val_predict'])
        test_predict = list(p['experiment'].metrics_results['test_predict'])
        shape_prevs =  len(train_predict) + len(val_predict) + len(test_predict)
        diff_real = shape_ts - shape_prevs
        df_actual = pd.concat([
                pd.DataFrame({'id': n, 'set':'tran', 'prev': diff_real * [None] + train_predict, 'model_name': model_name, 'serie_name': serie_name}),
                pd.DataFrame({'id': n, 'set':'val', 'prev': val_predict, 'model_name': model_name, 'serie_name': serie_name}),
                pd.DataFrame({'id': n, 'set':'test', 'prev': test_predict, 'model_name': model_name, 'serie_name': serie_name}),
        ])
        df_actual = df_actual.reset_index(drop=True).reset_index(drop=False)
        df_model= pd.concat(
            [
                df_actual,
                df_model
            
        ])
    
    if df_model['id'].nunique()==1:
        df_base = df_model.copy()
        df_model_temp = df_base.copy()
        for i in range(1, 10):
            df_model_temp['id'] = i
            df_model = pd.concat([df_model, df_model_temp])

    return df_model
    

for serie_name, df_info_serie in df_execs.groupby('serie_name'):
    df_real = input.load_raw_data(serie_name)
    shape_ts = df_real['y'].shape[0]
    df_model=pd.DataFrame()
    for dict_series in df_info_serie.to_dict('records'):
        model_name = dict_series['model_name']
        serie_name = dict_series['serie_name']
        pth = dict_series['pth']
        exec_pkl = generics.open_saved_result(pth)
    
        df_model = pd.concat(
             [open_execs(exec_pkl, model_name, serie_name, shape_ts),
              df_model
            ])
    df_model = df_model.pivot(index=['index', 'id', 'set'], columns='model_name', values= 'prev' )
    df_final = df_real.reset_index().merge(df_model.reset_index(), on='index', how='inner')

    df_final.sort_values(['id', 'index']).to_csv(f'{config.MODEL_DATA_PATH}{experiment_id}_{serie_name}.csv')

/tmp/ipykernel_26252/1219210896.py:10: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_actual = pd.concat([
/tmp/ipykernel_26252/1219210896.py:10: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_actual = pd.concat([
/tmp/ipykernel_26252/1219210896.py:10: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude